# 0. OpenAI API Function Calling (Advanced) — 제품 탑재형 AI 시나리오
**스마트홈/생활가전/로보틱스 탑재형 AI를 위한 Tool Orchestration**
- Responses API 기반 멀티-툴 루프
- Strict Schema(추론 안정성) + 에러 복구
- 위험 작업(초기화/펌웨어 업데이트 등) 가드(확인 질문)

---

## 🧩 시나리오(가상)
스마트홈 허브에는 LLM이 탑재되어 있고, 아래 기능을 수행합니다.

- 사용자 자연어 요청을 **도구 호출(function calling)** 로 변환해 기기를 제어
- 기기에서 올라오는 **이벤트/로그**를 조회해 원인을 추정
- 필요 시 **원격 진단/서비스 티켓**을 생성

실습에서는 네트워크나 실기기 없이, **더미 디바이스 상태/로그**를 메모리 상에서 다룹니다.


In [ ]:
!pip -q install -U openai pydantic

In [ ]:
import os, json, time, random, getpass
from typing import Optional, Literal, Any
from pydantic import BaseModel, Field
from openai import OpenAI

PROXY_URL = "http://10.0.1.5:8000/v1" 
PROXY_TOKEN = "" 


client = OpenAI(
    base_url=PROXY_URL,
    api_key=PROXY_TOKEN,
)

MODEL = os.getenv("OPENAI_MODEL", "gpt-4o-mini")  # 툴 플래닝은 비교적 큰 모델 권장


## 1) (더미) 디바이스 상태/로그/액션 함수 정의

실습에서는 실제 기기/모니터링 시스템 대신 **더미 데이터**로 흉내냅니다.  
중요한 것은 **함수의 스키마와 오케스트레이션 루프**입니다.


In [3]:
# -----------------------------
# 더미 디바이스(제품) 상태/이벤트 데이터
# -----------------------------
# - 실제 제품/현업 데이터가 아니라, 강의용 "가상" 데이터입니다.

DEVICES = {
    "vacuum": {
        "type": "robot_vacuum",
        "status": "docked",
        "battery": 0.78,
        "mode": "idle",
        "last_map": "living_room",
        "fw": "1.2.0",
    },
    "ac": {
        "type": "air_conditioner",
        "status": "online",
        "mode": "cool",
        "target_temp_c": 24,
        "humidity": 0.55,
        "fw": "3.5.1",
    },
    "washer": {
        "type": "washing_machine",
        "status": "online",
        "mode": "standby",
        "last_error": "5C",  # 배수 관련(예시)
        "fw": "2.1.7",
    },
    "fridge": {
        "type": "refrigerator",
        "status": "online",
        "mode": "normal",
        "door_open": False,
        "internal_temp_c": 3.0,
        "fw": "5.0.2",
    },
}

RECENT_EVENTS = {
    "vacuum": [
        "INFO 08:12:03 docked=true battery=0.78",
        "WARN 08:15:44 bump_detected count=12 room=living_room",
        "INFO 08:17:01 suction=normal",
    ],
    "ac": [
        "INFO 09:01:00 mode=cool target_temp=24",
        "INFO 09:05:10 power_w=680 humidity=0.55",
        "WARN 09:07:33 filter_warning=true",
    ],
    "washer": [
        "INFO 07:40:00 program=quick start",
        "ERROR 07:52:12 code=5C drain_timeout=true",
        "INFO 07:52:13 stopped=true water_level=mid",
    ],
    "fridge": [
        "INFO 10:10:00 internal_temp=3.0",
        "WARN 10:32:15 door_open=true duration_sec=95",
        "INFO 10:34:00 door_open=false",
    ],
}

SERVICE_TICKETS = []  # 생성된 티켓 저장(더미)

# -----------------------------
# Tool functions (모두 JSON string 반환)
# -----------------------------
def get_device_status(device: Literal["vacuum","ac","washer","fridge"]) -> str:
    """기기 상태/지표를 조회합니다."""
    if device not in DEVICES:
        return json.dumps({"ok": False, "error": "unknown_device"})
    return json.dumps({"ok": True, "device": device, **DEVICES[device]})

def get_recent_events(device: Literal["vacuum","ac","washer","fridge"], limit: int = 5) -> str:
    """최근 이벤트/로그 일부를 조회합니다."""
    if limit <= 0 or limit > 20:
        return json.dumps({"ok": False, "error": "invalid_limit", "limit": limit})
    events = RECENT_EVENTS.get(device, [])[-limit:]
    return json.dumps({"ok": True, "device": device, "limit": limit, "events": events})

def set_device_setting(device: Literal["vacuum","ac","washer","fridge"], setting: str, value: Any) -> str:
    """기기 설정을 변경합니다(비위험)."""
    if device not in DEVICES:
        return json.dumps({"ok": False, "error": "unknown_device"})
    # 아주 단순한 유효성 예시
    if setting == "target_temp_c":
        if device != "ac":
            return json.dumps({"ok": False, "error": "setting_not_supported", "device": device, "setting": setting})
        if not (16 <= float(value) <= 30):
            return json.dumps({"ok": False, "error": "out_of_range", "setting": setting, "min": 16, "max": 30})
        DEVICES[device]["target_temp_c"] = float(value)
    elif setting == "mode":
        # device별 허용 모드(예시)
        allowed = {
            "vacuum": ["idle", "clean", "dock"],
            "ac": ["cool", "dry", "fan", "off"],
            "washer": ["standby", "quick", "cotton", "rinse_spin", "off"],
            "fridge": ["normal", "power_cool", "vacation"],
        }
        if str(value) not in allowed.get(device, []):
            return json.dumps({"ok": False, "error": "invalid_mode", "device": device, "allowed": allowed.get(device, [])})
        DEVICES[device]["mode"] = str(value)
    else:
        return json.dumps({"ok": False, "error": "unknown_setting", "setting": setting})

    return json.dumps({"ok": True, "device": device, "updated": {setting: value}, "state": DEVICES[device]})

def run_remote_diagnosis(device: Literal["vacuum","ac","washer","fridge"]) -> str:
    """원격 자가진단(더미)"""
    if device not in DEVICES:
        return json.dumps({"ok": False, "error": "unknown_device"})
    # 장치별 간단 진단 결과(예시)
    if device == "washer":
        return json.dumps({"ok": True, "device": device, "diagnosis": {"code": "5C", "likely_cause": "배수 필터 막힘/배수 호스 꺾임"}})
    if device == "ac":
        return json.dumps({"ok": True, "device": device, "diagnosis": {"filter_warning": True, "recommendation": "필터 청소 후 재시도"}})
    return json.dumps({"ok": True, "device": device, "diagnosis": {"status": "no_critical_issue"}})

def update_firmware(device: Literal["vacuum","ac","washer","fridge"], target_version: str, confirmed: bool = False) -> str:
    """(위험) 펌웨어 업데이트. confirmed=True일 때만 수행."""
    if not confirmed:
        return json.dumps({"ok": False, "error": "confirmation_required", "message": "펌웨어 업데이트는 위험 작업입니다. confirmed=true로 다시 호출해야 합니다."})
    DEVICES[device]["fw"] = target_version
    return json.dumps({"ok": True, "device": device, "fw": DEVICES[device]["fw"]})

def factory_reset(device: Literal["vacuum","ac","washer","fridge"], confirmed: bool = False) -> str:
    """(매우 위험) 공장 초기화. confirmed=True일 때만 수행."""
    if not confirmed:
        return json.dumps({"ok": False, "error": "confirmation_required", "message": "공장 초기화는 매우 위험합니다. confirmed=true로 다시 호출해야 합니다."})
    # 아주 단순한 초기화 효과(예시)
    if device == "ac":
        DEVICES[device]["mode"] = "off"
        DEVICES[device]["target_temp_c"] = 24
    if device == "vacuum":
        DEVICES[device]["mode"] = "idle"
        DEVICES[device]["last_map"] = None
    return json.dumps({"ok": True, "device": device, "state": DEVICES[device]})

def create_service_ticket(device: Literal["vacuum","ac","washer","fridge"], summary: str, severity: str, evidence: list[str]) -> str:
    """서비스 티켓 생성(더미). 개인정보는 넣지 않습니다."""
    ticket = {
        "ticket_id": f"TCK-{1000 + len(SERVICE_TICKETS)}",
        "device": device,
        "summary": summary[:200],
        "severity": severity,
        "evidence": evidence[:10],
        "created_at": time.strftime("%Y-%m-%d %H:%M:%S"),
    }
    SERVICE_TICKETS.append(ticket)
    return json.dumps({"ok": True, "ticket": ticket})


## 2) 툴 스키마 정의 (`strict: true` + additionalProperties: false)

Responses API의 tool 정의는 보통 아래 필드를 사용합니다.
- `type`, `name`, `description`, `parameters`, `strict`

**실습 포인트:**  
- `additionalProperties: false`를 켜서 **예상치 못한 인자**가 오지 않게 만들고  
- `strict: true`로 **스키마 준수**를 강제합니다.


In [4]:
TOOLS = [
    {
        "type": "function",
        "name": "get_device_status",
        "description": "기기 상태/지표를 조회합니다. 상태 확인 질문에 사용.",
        "parameters": {
            "type": "object",
            "properties": {
                "device": {"type": "string", "enum": ["vacuum","ac","washer","fridge"]}
            },
            "required": ["device"],
            "additionalProperties": False,
        },
        "strict": True,
    },
    {
    "type": "function",
    "name": "get_recent_events",
    "description": "기기별 최근 이벤트/로그(더미)를 조회합니다. 원인 추정에 사용.",
    "parameters": {
        "type": "object",
        "properties": {
            "device": {"type": "string", "enum": ["vacuum","ac","washer","fridge"]},
            "limit": {"type": "integer", "minimum": 1, "maximum": 20},
        },
        "required": ["device", "limit"],
        "additionalProperties": False,
    },
    "strict": True
    },
    {
        "type": "function",
        "name": "set_device_setting",
        "description": "기기 설정을 변경합니다(예: 에어컨 온도/모드, 로봇청소기 모드). 위험하지 않은 변경에 사용.",
        "parameters": {
            "type": "object",
            "properties": {
                "device": {"type": "string", "enum": ["vacuum","ac","washer","fridge"]},
                "setting": {"type": "string", "enum": ["mode", "target_temp_c"]},
                "value": {"type": ["string","number"]},
            },
            "required": ["device","setting","value"],
            "additionalProperties": False,
        },
        "strict": True,
    },
    {
        "type": "function",
        "name": "run_remote_diagnosis",
        "description": "원격 자가진단(더미)을 실행하고 진단 결과를 반환합니다.",
        "parameters": {
            "type": "object",
            "properties": {
                "device": {"type": "string", "enum": ["vacuum","ac","washer","fridge"]}
            },
            "required": ["device"],
            "additionalProperties": False,
        },
        "strict": True,
    },
    {
        "type": "function",
        "name": "update_firmware",
        "description": "펌웨어 업데이트(위험 작업). 사용자 확인(confirmed=true) 후에만 실행하세요.",
        "parameters": {
            "type": "object",
            "properties": {
                "device": {"type": "string", "enum": ["vacuum","ac","washer","fridge"]},
                "target_version": {"type": "string"},
                "confirmed": {"type": "boolean", "default": False},
            },
            "required": ["device","target_version","confirmed"],
            "additionalProperties": False,
        },
        "strict": True,
    },
    {
        "type": "function",
        "name": "factory_reset",
        "description": "공장 초기화(매우 위험). 사용자 확인(confirmed=true) 후에만 실행하세요.",
        "parameters": {
            "type": "object",
            "properties": {
                "device": {"type": "string", "enum": ["vacuum","ac","washer","fridge"]},
                "confirmed": {"type": "boolean", "default": False},
            },
            "required": ["device","confirmed"],
            "additionalProperties": False,
        },
        "strict": True,
    },
    {
        "type": "function",
        "name": "create_service_ticket",
        "description": "A/S 또는 원격지원 티켓을 생성합니다. 개인정보는 포함하지 마세요.",
        "parameters": {
            "type": "object",
            "properties": {
                "device": {"type": "string", "enum": ["vacuum","ac","washer","fridge"]},
                "summary": {"type": "string"},
                "severity": {"type": "string", "enum": ["low","medium","high"]},
                "evidence": {"type": "array", "items": {"type": "string"}},
            },
            "required": ["device","summary","severity","evidence"],
            "additionalProperties": False,
        },
        "strict": True,
    },
]


## 3) 오케스트레이션 루프 구현 (핵심)

Responses API에서는 모델이 툴을 호출하면 `response.output` 안에 `function_call` 아이템이 들어옵니다.  
우리는 다음을 수행해야 합니다.

1) 모델 호출 → response 받기  
2) response.output를 입력 리스트에 누적  
3) function_call이 있으면 실제 파이썬 함수를 실행  
4) 결과를 `function_call_output` 아이템으로 다시 input에 넣기 (call_id 매칭)  
5) 모델을 다시 호출 → 최종 응답까지 반복

> **실습 난이도 포인트**  
> - call_id 매칭  
> - 여러 tool call 처리  
> - tool 실행 에러를 모델에게 전달  
> - 위험 작업 confirmation 가드


In [5]:
def dispatch_tool(name: str, arguments: dict) -> str:
    """tool name과 arguments를 받아 실제 함수를 호출하고 JSON string을 반환합니다."""
    try:
        if name == "get_device_status":
            return get_device_status(**arguments)
        if name == "get_recent_events":
            return get_recent_events(**arguments)
        if name == "set_device_setting":
            return set_device_setting(**arguments)
        if name == "run_remote_diagnosis":
            return run_remote_diagnosis(**arguments)
        if name == "update_firmware":
            return update_firmware(**arguments)
        if name == "factory_reset":
            return factory_reset(**arguments)
        if name == "create_service_ticket":
            return create_service_ticket(**arguments)

        return json.dumps({"ok": False, "error": "unknown_tool", "name": name})
    except Exception as e:
        # 에러를 모델이 이해할 수 있게 JSON으로 포장
        return json.dumps({"ok": False, "error": "exception", "message": str(e), "tool": name})

def run_tool_loop(user_message: str, *, allow_destructive: bool = False, max_rounds: int = 6) -> dict:
    """멀티-툴 루프를 실행하고 최종 응답/로그를 반환합니다."""
    instructions = """당신은 '스마트홈/생활가전/로보틱스' 제품 탑재형 AI 어시스턴트입니다.
- 상태 확인/원인 추정에는 get_device_status, get_recent_events, run_remote_diagnosis를 사용하세요.
- 설정 변경은 set_device_setting을 사용하세요.
- update_firmware, factory_reset는 위험 작업입니다.
  - allow_destructive=false이면 절대 confirmed=true로 호출하지 말고, 사용자에게 확인 질문을 먼저 하세요.
  - allow_destructive=true일 때만 confirmed=true로 호출을 고려하세요.
- create_service_ticket는 필요 시 생성하되, 개인정보/민감정보는 절대 포함하지 마세요.
- 최종 답변은 한국어로, (1)현황 (2)근거 (3)권장조치 순서로 작성하세요.
"""

    input_list: list[dict[str, Any]] = [
        {"role": "user", "content": user_message}
    ]

    trace = []

    for round_idx in range(1, max_rounds + 1):
        resp = client.responses.create(
            model=MODEL,
            instructions=instructions,
            tools=TOOLS,
            input=input_list,
        )

        trace.append({"round": round_idx, "raw_output_count": len(resp.output)})
        # 모델이 만든 아이템을 input에 누적
        input_list += resp.output

        # tool call 수집
        tool_calls = [item for item in resp.output if getattr(item, "type", None) == "function_call"]
        if not tool_calls:
            return {
                "ok": True,
                "final_text": resp.output_text,
                "trace": trace,
                "input_list": input_list,
            }

        # tool call 처리
        for call in tool_calls:
            args = json.loads(call.arguments) if isinstance(call.arguments, str) else call.arguments

            # 위험 작업 가드(루프 레벨)
            if call.name in {"update_firmware", "factory_reset"} and not allow_destructive:
                output = json.dumps({
                    "ok": False,
                    "error": "destructive_not_allowed",
                    "message": "현재 설정(allow_destructive=false)에서는 위험 작업을 실행할 수 없습니다. 사용자 확인을 받으세요.",
                    "tool": call.name,
                })
            else:
                output = dispatch_tool(call.name, args)

            input_list.append({
                "type": "function_call_output",
                "call_id": call.call_id,
                "output": output,
            })

    return {
        "ok": False,
        "error": "max_rounds_exceeded",
        "trace": trace,
        "input_list": input_list,
    }

# 예시 실행
# result = run_tool_loop("세탁기 5C 에러가 떠요. 원인 추정하고 지금 할 수 있는 조치 알려줘.", allow_destructive=False)
# print(result["final_text"])


In [6]:
result = run_tool_loop("세탁기 5C 에러가 떠요. 원인 추정하고 지금 할 수 있는 조치 알려줘.", allow_destructive=False)
print(result["final_text"])

(1) **현황**  
세탁기에서 5C 에러가 발생했습니다. 현재 상태는 온라인이며 대기 중입니다. 최근 이벤트 기록에 따르면 배수 타임아웃 문제가 발생하여 에러가 발생했음을 알 수 있습니다.

(2) **근거**  
- 에러 코드는 5C로, 이는 배수 문제와 관련이 있습니다.
- 최근 이벤트 기록에서 "ERROR 5C drain_timeout=true"가 확인되었습니다.
- 원격 진단 결과 배수 호스 막힘 또는 배수 펌프 고장 가능성이 제기되었습니다.

(3) **권장조치**  
1. 세탁기의 배수 호스를 확인하여 막힘이 있는지 점검하세요.  
2. 배수 호스를 청소하거나 필요시 교체하세요.  
3. 배수 펌프에 문제가 있을 경우 서비스 팀에 문의하여 점검 받는 것을 권장합니다.  
4. 만약 문제가 지속된다면 서비스 티켓을 생성해 지원 요청을 할 수 있습니다.


In [7]:
result

{'ok': True,
 'final_text': '(1) **현황**  \n세탁기에서 5C 에러가 발생했습니다. 현재 상태는 온라인이며 대기 중입니다. 최근 이벤트 기록에 따르면 배수 타임아웃 문제가 발생하여 에러가 발생했음을 알 수 있습니다.\n\n(2) **근거**  \n- 에러 코드는 5C로, 이는 배수 문제와 관련이 있습니다.\n- 최근 이벤트 기록에서 "ERROR 5C drain_timeout=true"가 확인되었습니다.\n- 원격 진단 결과 배수 호스 막힘 또는 배수 펌프 고장 가능성이 제기되었습니다.\n\n(3) **권장조치**  \n1. 세탁기의 배수 호스를 확인하여 막힘이 있는지 점검하세요.  \n2. 배수 호스를 청소하거나 필요시 교체하세요.  \n3. 배수 펌프에 문제가 있을 경우 서비스 팀에 문의하여 점검 받는 것을 권장합니다.  \n4. 만약 문제가 지속된다면 서비스 티켓을 생성해 지원 요청을 할 수 있습니다.',
 'trace': [{'round': 1, 'raw_output_count': 3},
  {'round': 2, 'raw_output_count': 1}],
 'input_list': [{'role': 'user',
   'content': '세탁기 5C 에러가 떠요. 원인 추정하고 지금 할 수 있는 조치 알려줘.'},
  ResponseFunctionToolCall(arguments='{"device":"washer"}', call_id='call_AR8UUD0NwGKheNxn3TynhF3U', name='get_device_status', type='function_call', id='fc_0ab26f029dfe42c70069b8321610808195b5618e00d5f588f9', status='completed'),
  ResponseFunctionToolCall(arguments='{"device":"washer","limit":5}', call_id='call_cVfsRWUIHzTUCFxNp02c5pnc', nam

## 4) (업그레이드) 최종 결과를 Device Support Report로 구조화하기

위 `run_tool_loop()`는 자연어 답변을 반환하지만,
운영에서는 **JSON/스키마 기반 리포트**가 더 유용할 때가 많습니다.

여기서는 한 번 더 `responses.parse`로 **최종 요약을 구조화**합니다.


In [8]:
class DeviceSupportReport(BaseModel):
    title: str = Field(description="이슈 요약 제목")
    severity: Literal["low","medium","high"] = Field(description="심각도")
    device: Literal["vacuum","ac","washer","fridge"] = Field(description="대상 기기")
    evidence: list[str] = Field(description="근거(로그/진단 결과 등)")
    recommended_actions: list[str] = Field(description="권장 조치(사용자/기사/원격 포함)")
    safe_next_question: Optional[str] = Field(default=None, description="사용자 확인/추가정보가 필요하면 질문")


def normalize_item(item):
    if isinstance(item, dict):
        return item

    if hasattr(item, "model_dump"):
        return item.model_dump()

    if hasattr(item, "__dict__"):
        return item.__dict__

    return str(item)


def summarize_to_report(conversation_items: list[Any]) -> DeviceSupportReport:
    prompt = """지금까지의 대화/툴 결과를 바탕으로 '제품 지원 리포트'를 작성하세요.
- severity는 low/medium/high 중 하나
- evidence에는 툴 결과에서 나온 핵심 근거를 3~6개 bullet로 정리
- recommended_actions는 (사용자가 할 일)과 (원격/기사 대응)으로 나눠 3~6개 제안
- 개인정보/민감정보는 포함하지 마세요.
"""

    normalized_items = [normalize_item(item) for item in conversation_items]

    resp = client.responses.parse(
        model=MODEL,
        input=[
            {"role": "system", "content": prompt},
            {"role": "user", "content": json.dumps(normalized_items, ensure_ascii=False)},
        ],
        text_format=DeviceSupportReport,
    )
    return resp.output_parsed


In [9]:
result = run_tool_loop(
    "세탁기 5C 에러가 떠요. 원인 추정하고 지금 할 수 있는 조치 알려줘.",
    allow_destructive=False,
)


report = summarize_to_report(result["input_list"])
print(report)

title='DeviceSupportReport' severity='medium' device='washer' evidence=['5C 에러가 발생함.', '배수 타임아웃으로 인한 문제로 확인됨.', '최근 이벤트 로그에서 배수 문제 발생 확인됨.', '진단 결과 배수 관련 문제가 원인으로 확인됨.', '세탁기는 현재 온라인 상태임.'] recommended_actions=['세탁기의 배수 호스와 필터를 점검하여 막힘이 있는지 확인합니다.', '배수 호스가 꼬이거나 엉켜 있지 않은지 확인합니다.', '문제 해결 후 세탁기를 재가동하여 정상 작동하는지 확인합니다.'] safe_next_question=None


## 📝 실습 과제 (To-Do)

### 문제 1) 툴 1개 추가 (난이도: 중)
- `get_latency_slo(server)` 툴을 추가하세요.
  - 반환: `{"ok": true, "server": ..., "slo_ms": ..., "p95_ms": ...}`
  - `TOOLS`에도 스키마 추가

### 문제 2) 안전 가드 강화 (난이도: 상)
- restart_service는 `confirmed=true`여도 **추가 조건**이 필요하도록 만드세요.
  - 예: `server="db"`인 경우에만 허용, 또는 `reason` 필수

### 문제 3) 자동 티켓 생성 플로우 (난이도: 상)
- 사용자가 "티켓 만들어줘"라고 하면,
  - 모델이 로그/상태를 모으고
  - `create_incident_ticket`를 호출한 뒤
  - ticket_id를 포함한 최종 답변을 하도록 유도하세요.

> 아래 셀에 TODO로 구현하고, 마지막에 테스트 질문 2~3개를 던져 동작을 확인하세요.


In [10]:
# ✅ TODO 셀
# 1) get_warranty_status(device) 도구를 구현하세요. (더미 DB 기반)
# 2) TOOLS에 get_warranty_status 스키마를 추가하세요. (strict + additionalProperties: false)
# 3) run_tool_loop 가드 강화:
#    - allow_destructive=false일 때 위험 작업을 호출하면, 모델이 '확인 질문'을 하도록 유도 메시지를 더 친절히 작성해보세요.

raise NotImplementedError("학생용 TODO 셀입니다. 아래 '참고 답안'을 보기 전까지 구현해보세요.")


NotImplementedError: 학생용 TODO 셀입니다. 아래 '참고 답안'을 보기 전까지 구현해보세요.

In [ ]:
# ✅ 참고 답안

WARRANTY = {
    "vacuum": {"in_warranty": True, "days_left": 120},
    "ac": {"in_warranty": True, "days_left": 420},
    "washer": {"in_warranty": False, "days_left": 0},
    "fridge": {"in_warranty": True, "days_left": 30},
}

def get_warranty_status(device: Literal["vacuum","ac","washer","fridge"]) -> str:
    """기기 보증 상태(더미)를 반환합니다."""
    w = WARRANTY.get(device)
    if not w:
        return json.dumps({"ok": False, "error": "unknown_device"})
    return json.dumps({"ok": True, "device": device, **w})

# dispatch_tool에 분기 추가(실습에서는 직접 수정)
old_dispatch = dispatch_tool

def dispatch_tool(name: str, arguments: dict) -> str:
    try:
        if name == "get_warranty_status":
            return get_warranty_status(**arguments)
        return old_dispatch(name, arguments)
    except Exception as e:
        return json.dumps({"ok": False, "error": "exception", "message": str(e), "tool": name})

# TOOLS에 스키마 추가
TOOLS.append({
    "type": "function",
    "name": "get_warranty_status",
    "description": "기기 보증 상태(더미)를 조회합니다. A/S 안내에 사용.",
    "parameters": {
        "type": "object",
        "properties": {
            "device": {"type": "string", "enum": ["vacuum","ac","washer","fridge"]}
        },
        "required": ["device"],
        "additionalProperties": False,
    },
    "strict": True,
})

# 예시 실행
# result = run_tool_loop("세탁기 5C 에러가 떠요. 보증기간도 확인해주고, 티켓 생성까지 해줘.", allow_destructive=False)
# print(result["final_text"])


In [ ]:
result = run_tool_loop("세탁기 5C 에러가 떠요. 보증기간도 확인해주고, 티켓 생성까지 해줘.", allow_destructive=False)
print(result["final_text"])

(1) **현황**: 세탁기에 5C 에러 코드가 발생했습니다. 이는 배수 불량과 관련된 문제입니다. 현재 제품은 보증 기간이 만료된 상태입니다.

(2) **근거**: 최근 이벤트 로그에서 "ERROR 07:52:12 code=5C drain_timeout=true"가 기록되어 있고, 이로 인해 세탁이 중단되었습니다. 보증 조회 결과, 보증 기간이 종료되었습니다.

(3) **권장 조치**: 배수 호스의 막힘 여부를 확인하고 청소해보세요. 문제가 지속되면 전문 수리기사의 점검이 필요합니다. 서비스 티켓 [TCK-1000](#)을 생성했으니, 수리 요청을 진행하시길 바랍니다.
